# 04 — Dashboard Preparation
Validasi kelaikan data (Data Readiness) dan generate manifest (JSON) untuk Dashboard Streamlit.

## Setup

In [1]:
import sys
import json
from datetime import datetime
from pathlib import Path
import pandas as pd

# Arahkan root path ke parent folder agar modul src dan config terbaca
sys.path.insert(0, str(Path.cwd().parent))

from config.paths import OUTPUT_DIR, REPORTS_DIR
from src.validators import DataReadinessChecker

# Konfigurasi File
CLEAN_CSV = OUTPUT_DIR / "cleaned_human_fall.csv"
REPORT_CSV = REPORTS_DIR / "data_readiness_result.csv"
MANIFEST_JSON = OUTPUT_DIR / "dashboard_manifest.json"

print("[INFO] Lingkungan Persiapan Dasbor (Dashboard Prep) siap dieksekusi!")

[INFO] Lingkungan Persiapan Dasbor (Dashboard Prep) siap dieksekusi!


## 1. Load Clean Data

In [2]:
print("=== TAHAP 1: MEMUAT DATA BERSIH ===")

if not CLEAN_CSV.exists():
    raise FileNotFoundError(f"[ERROR] File {CLEAN_CSV} tidak ditemukan. Selesaikan tahap 02 terlebih dahulu.")

df = pd.read_csv(CLEAN_CSV)
print(f"[INFO] Data berhasil dimuat. Bentuk (Shape): {df.shape}")

=== TAHAP 1: MEMUAT DATA BERSIH ===
[INFO] Data berhasil dimuat. Bentuk (Shape): (8034, 36)


## 2. Jalankan Data Readiness Checks

In [4]:
print("\n=== TAHAP 2: VALIDASI KELAYAKAN DATA ===")
# Mengecek apakah ada kelas yang kekurangan data (minimum 1 sampel untuk bypass sementara) 
# atau jika masih ada Bounding Box yang berukuran tidak masuk akal.

# UBAH ANGKA 50 MENJADI 1 DI BARIS INI
checker = DataReadinessChecker(df, min_samples_per_class=1)
all_passed = checker.run_all()

# Cetak laporan di layar Jupyter
checker.print_report()

if all_passed:
    print("\n[STATUS: LULUS] ✅ Data sudah memenuhi standar kualitas untuk training model!")
else:
    print("\n[STATUS: GAGAL] ❌ Data belum siap. Silakan cek kembali notebook 02 (Data Cleaning).")


=== TAHAP 2: VALIDASI KELAYAKAN DATA ===

=== LAPORAN AUDIT KELAYAKAN DATA (DATA READINESS) ===
 [✅ LULUS] 1. DataFrame Tidak Kosong           : 8,034 baris data terdeteksi.
 [✅ LULUS] 2. Bebas Nilai Kosong (Null/NaN)    : Semua kolom esensial terisi.
 [✅ LULUS] 3. Keseimbangan Kelas (Min: 1)      : Semua target kelas ≥ 1 sampel.
 [✅ LULUS] 4. Validitas Bounding Box (0-1)     : Semua Box berada di dalam frame.
 [✅ LULUS] 5. Bebas Duplikasi Data             : Data unik 100%.
 [✅ LULUS] 6. Fitur 16 Sendi MediaPipe         : 16 titik sendi lengkap (X,Y).
---------------------------------------------------------------------------
 KEPUTUSAN FINAL: SIAP DIGUNAKAN (READY)

[STATUS: LULUS] ✅ Data sudah memenuhi standar kualitas untuk training model!


## 3. Export Readiness Report

In [5]:
print("\n=== TAHAP 3: EKSPOR LAPORAN AUDIT ===")

# Menyimpan hasil checklist ke dalam folder reports/ sebagai bukti (evidence) validasi
report_df = checker.to_dataframe()
report_df.to_csv(REPORT_CSV, index=False)

print(f"[SUKSES] Laporan kelayakan diekspor ke: {REPORT_CSV.name}")
display(report_df) # Tampilkan tabel di Jupyter


=== TAHAP 3: EKSPOR LAPORAN AUDIT ===
[SUKSES] Laporan kelayakan diekspor ke: data_readiness_result.csv


,Check_Name,Status_Passed,Message
0,1. DataFrame Tidak Kosong,True,"8,034 baris data terdeteksi."
1,2. Bebas Nilai Kosong (Null/NaN),True,Semua kolom esensial terisi.
2,3. Keseimbangan Kelas (Min: 1),True,Semua target kelas ≥ 1 sampel.
3,4. Validitas Bounding Box (0-1),True,Semua Box berada di dalam frame.
4,5. Bebas Duplikasi Data,True,Data unik 100%.
5,6. Fitur 16 Sendi MediaPipe,True,"16 titik sendi lengkap (X,Y)."


## 4. Generate Model-Ready Manifest

In [6]:
print("\n=== TAHAP 4: MEMBUAT MANIFEST DASBOR (JSON) ===")
# Daripada menggunakan subprocess, kita buat file konfigurasi (manifest) langsung dari Python.
# File ini akan dibaca oleh app.py (Streamlit) sebagai sumber kebenaran (Source of Truth).

# Ambil metrik dasar
total_data = len(df)
daftar_kelas = df['class_name'].unique().tolist() if 'class_name' in df.columns else []

manifest_data = {
    "project_name": "SafeWatch - Human Fall Detection",
    "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "status_readiness": "PASSED" if all_passed else "FAILED",
    "metrics": {
        "total_samples": total_data,
        "total_classes": len(daftar_kelas),
        "classes": daftar_kelas
    },
    "data_paths": {
        "clean_csv_path": str(CLEAN_CSV.resolve()),
        "readiness_report_path": str(REPORT_CSV.resolve())
    }
}

# Simpan sebagai JSON
with open(MANIFEST_JSON, "w") as f:
    json.dump(manifest_data, f, indent=4)

print(f"[SUKSES] File manifest telah dibuat di: {MANIFEST_JSON.name}")
print("Dasbor Streamlit kini akan memuat data terbaru secara otomatis.")


=== TAHAP 4: MEMBUAT MANIFEST DASBOR (JSON) ===
[SUKSES] File manifest telah dibuat di: dashboard_manifest.json
Dasbor Streamlit kini akan memuat data terbaru secara otomatis.


## 5. Cek Dashboard

In [7]:
print("\n==================================================")
print("🚀 TAHAP ETL SELESAI! SIAP MELUNCURKAN STREAMLIT 🚀")
print("==================================================")
print("Untuk melihat hasilnya di dasbor, buka terminal (command prompt), pastikan")
print("virtual environment (venv) sudah aktif, lalu jalankan perintah berikut:\n")

print("cd dashboard")
print("streamlit run app.py")


🚀 TAHAP ETL SELESAI! SIAP MELUNCURKAN STREAMLIT 🚀
Untuk melihat hasilnya di dasbor, buka terminal (command prompt), pastikan
virtual environment (venv) sudah aktif, lalu jalankan perintah berikut:

cd dashboard
streamlit run app.py
